# **Retail Sales Project**
## Business Requirements:
we have superstore sample retail data and our stakeholder wants to track the below Business metriks for
1. How many total number of customers we have ?
2. total number of orders we have
3. Total number of sales as of now
4. Total profit
5. Top sales by country
6. Most profitable region and country
7. Top sales category products
8. Top 10 sales sub category products
9. Most ordered quantity product
10. Top customer based on sales, city

**Note: dashboard/Notebook should be refresh based on weekly and monthly basis**

# AWS Part

## 1️⃣ Data Storage Setup

You created a bucket in **Amazon S3** to store your data.

You organized it into two stages:

**Raw data**

```
s3://superstore-raw-data-input/
```

**Processed data**

```
s3://superstore-processed-data/output/
```

Purpose:

* Raw CSV files are uploaded here
* Processed cleaned data is saved after ETL

---

# 2️⃣ ETL Pipeline Creation

You created an ETL job in **AWS Glue**.

This job performs:

### Data Processing Steps

* Reads CSV files from S3
* Cleans corrupted rows
* Fixes column names
* Joins two datasets
* Converts date formats
* Standardizes category names
* Writes cleaned data back to S3

This is your **ETL (Extract → Transform → Load) pipeline**.

---

# 3️⃣ Automation with Serverless Function

You created a trigger using **AWS Lambda**.

Lambda performs:

1. Detects when a new CSV file is uploaded
2. Checks if the file is `.csv`
3. Starts the Glue ETL job automatically

Example action:

```
Upload CSV → Lambda runs → Glue ETL starts
```

---

# 4️⃣ Event-Based Trigger Setup

You configured **Amazon S3** event notifications.

Trigger condition:

```
Object Created
Suffix = .csv
```

Meaning:

Whenever a new CSV is uploaded → Lambda is triggered.

---

# 5️⃣ Logging and Monitoring

Logs are stored in **Amazon CloudWatch**.

This helps you:

* Debug Lambda
* Check ETL job execution
* Monitor pipeline errors

---

# Final Pipeline Architecture

Your system now works like this:

```
CSV Upload
     │
     ▼
Amazon S3 (Raw Data)
     │
     ▼
AWS Lambda Trigger
     │
     ▼
AWS Glue ETL
     │
     ▼
Amazon S3 (Processed Data)
     │
     ▼
Databricks Notebook
     │
     ▼
Analytics / Visualization
```

This is a **serverless data pipeline**.

---

**Collecting AWS S3 processed data and migrating it in Databricks Notebook**

In [0]:
dbutils.widgets.dropdown("timeperiod", "Weekly", ["Weekly", "Mounthly"])

In [0]:
from datetime import date, timedelta, datetime
from pyspark.sql.functions import *

# Create the widget first
dbutils.widgets.text("time_period", "Monthly", "Time Period (Weekly/Monthly)")

# Then get the value
time_period = dbutils.widgets.get("time_period")
print(f"Time period selected: {time_period}")

today = date.today()

if time_period == 'Weekly':
    start_date = today - timedelta(days=today.weekday(), weeks=1) - timedelta(days=1)
    end_date = start_date + timedelta(days=6)
else:  # Monthly or any other value defaults to Monthly
    first = today.replace(day=1)
    end_date = first - timedelta(days=1)
    start_date = first - timedelta(days=end_date.day)
    
print(f"Start date: {start_date}, End date: {end_date}")

In [0]:
df = spark.read.csv("/Volumes/workspace/default/apacha-data/project data/superstore_processed_data.csv", header=True, inferSchema=True)

display(df)

In [0]:
df.createOrReplaceTempView("sample")

In [0]:
last_order_date = df.agg(max("order_date")).collect()[0][0]

print(f"Last order date: {last_order_date}")

In [0]:
%sql
select count(distinct Customer_id) from sample

In [0]:
display(spark.sql(f"""select count(distinct Customer_id) from sample where Order_Date between '01-June-2021' and '10-June-2022'"""))

In [0]:
%sql
select count(distinct Order_id) from sample

In [0]:
display(spark.sql(f"""select count(distinct Order_id) from sample where Order_Date between '24-October-2024' and '31-October-2024'"""))

In [0]:
%sql
select sum(Sales), sum(Profit), max(Sales), max(Profit) from sample

In [0]:
%sql
select sum(Sales), Country from sample group by 2

In [0]:
%sql
select sum(Sales), Country, Region from sample group by 2, 3 order by 1 desc

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
select sum(Sales), Category from sample group by 2 order by 1 desc

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
select sum(Sales), Sub_Category from sample group by 2 order by 1 desc limit 10

In [0]:
%sql
select sum(Sales), Sub_Category from sample group by 2 order by 1 desc

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
select sum(Quantity), Product_name from sample group by 2 order by 1 desc

In [0]:
%sql
select sum(Sales), Customer_name, City from sample group by 2, 3 order by 1 desc